# 🛡️ Credit Card Fraud Detection System
## Comprehensive End-to-End Machine Learning Pipeline

**Author:** Production ML Engineering Team  
**Target Audience:** FinTech Teams, Fraud Analysts, & Risk Operations  
**Goal:** Develop an interpretable, production-ready fraud classification model under extreme class imbalance (0.172% fraud).

--- 
### 1. Environment Setup & Imports
We import foundational numerical, modeling, and visualization libraries. Warnings are suppressed for clean reporting.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, f1_score, precision_score, recall_score
)

import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Add src to path to enable custom modules
sys.path.append(os.path.abspath('../src'))
from data_preprocessing import DataPreprocessor, generate_synthetic_data
from feature_engineering import FeatureEngineer

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('✓ Environment configured and libraries loaded successfully.')

--- 
### 2. Dataset Loading & Inspection
We load the credit card transaction dataset. If the dataset does not exist locally in `../data/creditcard.csv`, we generate a synthetic dataset conforming to the identical Kaggle ULB distribution.

In [ ]:
data_path = os.path.join('..', 'data', 'creditcard.csv')

if not os.path.exists(data_path):
    print("Dataset not found locally. Generating synthetic Kaggle ULB benchmark replica...")
    df = generate_synthetic_data(n_samples=284807, fraud_ratio=0.00173, output_path=data_path)
else:
    print(f"Loading dataset from {data_path}...")
    df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
display(df.head())

--- 
### 3. Exploratory Data Analysis (EDA)
In financial fraud detection, class imbalance is severe. Visualizing the proportion of legitimate to fraudulent events underscores why traditional accuracy is a deceptive metric.

In [ ]:
class_counts = df['Class'].value_counts()
fraud_rate = df['Class'].mean() * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
sns.barplot(x=['Legitimate (0)', 'Fraud (1)'], y=class_counts.values, palette=['#4CAF50', '#F44336'], ax=ax[0])
ax[0].set_yscale('log')
ax[0].set_title('Class Distribution (Log Scale)', fontsize=14, fontweight='bold')
ax[0].set_ylabel('Transaction Count (log)')
for i, v in enumerate(class_counts.values):
    ax[0].text(i, v * 1.2, f"{v:,} ({v/len(df)*100:.3f}%)", ha='center', fontweight='bold')

# Donut chart
ax[1].pie(class_counts.values, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%',
          colors=['#4CAF50', '#F44336'], explode=(0, 0.15), startangle=90,
          wedgeprops=dict(width=0.4, edgecolor='w'))
ax[1].set_title(f'Fraud Prevalence: {fraud_rate:.4f}%', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

#### 3.1 Transaction Amount Analysis
Let us compare transaction amount statistics between legitimate and fraudulent transactions.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df[df['Class'] == 0]['Amount'].clip(upper=500), bins=50, color='#4CAF50', kde=True, ax=ax[0])
ax[0].set_title('Legitimate Transactions Amount (Capped at $500)', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Amount ($)')

sns.histplot(df[df['Class'] == 1]['Amount'].clip(upper=500), bins=50, color='#F44336', kde=True, ax=ax[1])
ax[1].set_title('Fraudulent Transactions Amount (Capped at $500)', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Amount ($)')

plt.tight_layout()
plt.show()

print("Amount Descriptive Statistics:")
display(df.groupby('Class')['Amount'].describe().rename(index={0: 'Legitimate', 1: 'Fraud'}))

#### 3.2 Time-Based Transaction Distribution
Analyzing transaction velocity by hour reveals operational patterns and off-peak vulnerabilities.

In [ ]:
df_time = df.copy()
df_time['Hour'] = (df_time['Time'] / 3600).astype(int) % 24

hourly_stats = df_time.groupby(['Hour', 'Class']).size().unstack(fill_value=0)
hourly_fraud_rate = (hourly_stats[1] / (hourly_stats[0] + hourly_stats[1])) * 100

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.plot(hourly_stats.index, hourly_stats[0], color='#4CAF50', lw=2, marker='o', label='Legitimate Volume')
ax2.plot(hourly_stats.index, hourly_fraud_rate, color='#F44336', lw=2.5, marker='s', linestyle='--', label='Fraud Rate (%)')

ax1.set_xlabel('Hour of Day (0-23)', fontsize=12)
ax1.set_ylabel('Legitimate Volume', color='#4CAF50', fontsize=12)
ax2.set_ylabel('Fraud Rate (%)', color='#F44336', fontsize=12)
plt.title('Transaction Volume and Fraud Probability by Hour of Day', fontsize=14, fontweight='bold')
plt.xticks(range(0, 24))
plt.show()

#### 3.3 Correlation Analysis of PCA Features with Fraud

In [ ]:
corrs = df.corr()['Class'].drop('Class').sort_values()
top_corrs = pd.concat([corrs.head(6), corrs.tail(6)])

plt.figure(figsize=(12, 5))
colors = ['#F44336' if x < 0 else '#2196F3' for x in top_corrs.values]
sns.barplot(x=top_corrs.index, y=top_corrs.values, palette=colors)
plt.title('Top Negative and Positive Correlated Features with Class', fontsize=14, fontweight='bold')
plt.ylabel('Correlation with Fraud')
plt.axhline(0, color='black', lw=0.8, linestyle='--')
plt.show()

--- 
### 4. Data Preprocessing & Splitting (Leakage Prevention)
To strictly avoid data leakage:
1. We perform a stratified train/validation/test split (70% / 10% / 20%).
2. We fit `RobustScaler` on `Time` and `Amount` exclusively on the training set.
3. We transform validation and test sets using the fitted scaler parameters.

In [ ]:
preprocessor = DataPreprocessor(random_state=42)
clean_df = preprocessor.clean_data(df)
X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.split_data(clean_df, test_size=0.2, val_size=0.1)

X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)

print(f"✓ Data Preprocessing Complete:")
print(f"  Training:   {X_train_scaled.shape[0]:,} samples ({y_train.sum()} frauds)")
print(f"  Validation: {X_val_scaled.shape[0]:,} samples ({y_val.sum()} frauds)")
print(f"  Testing:    {X_test_scaled.shape[0]:,} samples ({y_test.sum()} frauds)")

--- 
### 5. Feature Engineering
We engineer cyclical time representations, amount statistics, and PCA interaction signals.

In [ ]:
fe = FeatureEngineer()
X_train_fe = fe.fit_transform(X_train_scaled)
X_val_fe = fe.transform(X_val_scaled)
X_test_fe = fe.transform(X_test_scaled)

print(f"Features before engineering: {X_train_scaled.shape[1]}")
print(f"Features after engineering:  {X_train_fe.shape[1]}")
new_cols = [c for c in X_train_fe.columns if c not in X_train_scaled.columns]
print(f"New features added: {new_cols}")

--- 
### 6. Handling Class Imbalance
We test three imbalance strategies applied strictly on `X_train_fe`:
1. **Cost-Sensitive Class Weighting** (`class_weight='balanced'`)
2. **SMOTE Oversampling** (synthetic fraud instances)
3. **Random Undersampling** (balanced majority instances)

In [ ]:
# SMOTE resampled training set
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_fe, y_train)

# Undersampled training set
rus = RandomUnderSampler(random_state=42, sampling_strategy=0.5)
X_train_rus, y_train_rus = rus.fit_resample(X_train_fe, y_train)

print("Training Set Sizes Across Strategies:")
print(f"  Original / Class Weighted: {len(X_train_fe):,} samples ({y_train.sum()} fraud)")
print(f"  SMOTE Oversampled:         {len(X_train_smote):,} samples ({y_train_smote.sum()} fraud)")
print(f"  Random Undersampled:       {len(X_train_rus):,} samples ({y_train_rus.sum()} fraud)")

--- 
### 7. Multi-Model Training & Benchmarking
We train and benchmark Logistic Regression, Random Forest, XGBoost, and LightGBM across all strategies.

In [ ]:
models = {
    'Logistic Regression (Weighted)': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest (Weighted)': RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=15, n_jobs=-1, random_state=42),
    'XGBoost (Weighted)': xgb.XGBClassifier(scale_pos_weight=100, n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42, eval_metric='logloss'),
    'LightGBM (Weighted)': lgb.LGBMClassifier(is_unbalance=True, n_estimators=150, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1),
    'XGBoost (SMOTE)': xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42, eval_metric='logloss'),
    'Random Forest (Undersampled)': RandomForestClassifier(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
}

results = []
fitted_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    if 'SMOTE' in name:
        model.fit(X_train_smote, y_train_smote)
    elif 'Undersampled' in name:
        model.fit(X_train_rus, y_train_rus)
    else:
        model.fit(X_train_fe, y_train)
        
    fitted_models[name] = model
    
    # Validation evaluations
    val_pred = model.predict(X_val_fe)
    val_proba = model.predict_proba(X_val_fe)[:, 1]
    
    results.append({
        'Model': name,
        'Precision': precision_score(y_val, val_pred, zero_division=0),
        'Recall': recall_score(y_val, val_pred, zero_division=0),
        'F1-Score': f1_score(y_val, val_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_val, val_proba),
        'PR-AUC': average_precision_score(y_val, val_proba)
    })

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
display(results_df.style.highlight_max(subset=['Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC'], color='#c8e6c9'))

--- 
### 8. Test Set Evaluation & Diagnostics
We select the champion model and evaluate it against the unseen Test Set.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
champion_model = fitted_models[best_model_name]
print(f"Selected Champion Model: {best_model_name}")

y_test_pred = champion_model.predict(X_test_fe)
y_test_proba = champion_model.predict_proba(X_test_fe)[:, 1]

print("\n--- Test Classification Report ---")
print(classification_report(y_test, y_test_pred, target_names=['Legitimate', 'Fraud']))

#### 8.1 Confusion Matrix, ROC, and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'])
axes[0].set_title(f'Confusion Matrix\n{best_model_name}', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
axes[1].plot(fpr, tpr, color='#2196F3', lw=2, label=f'AUC = {roc_auc_score(y_test, y_test_proba):.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

# 3. Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_test_proba)
axes[2].plot(rec, prec, color='#4CAF50', lw=2, label=f'PR-AUC = {average_precision_score(y_test, y_test_proba):.4f}')
axes[2].axhline(y_test.mean(), color='gray', linestyle='--', label=f'Baseline ({y_test.mean():.4f})')
axes[2].set_title('Precision-Recall Curve', fontweight='bold')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

#### 8.2 Decision Threshold Optimization
We visualize the trade-off between Precision and Recall across various probability cutoffs.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 30)
precs, recs, f1s = [], [], []

for t in thresholds:
    preds_t = (y_test_proba >= t).astype(int)
    precs.append(precision_score(y_test, preds_t, zero_division=0))
    recs.append(recall_score(y_test, preds_t, zero_division=0))
    f1s.append(f1_score(y_test, preds_t, zero_division=0))

best_idx = np.argmax(f1s)
best_t = thresholds[best_idx]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, precs, label='Precision', color='#2196F3', lw=2)
plt.plot(thresholds, recs, label='Recall', color='#F44336', lw=2)
plt.plot(thresholds, f1s, label='F1-Score', color='#4CAF50', lw=2.5, linestyle='--')
plt.axvline(best_t, color='purple', linestyle=':', label=f'Optimal F1 Threshold ({best_t:.2f})')
plt.title('Threshold Analysis (Precision vs. Recall vs. F1)', fontsize=14, fontweight='bold')
plt.xlabel('Probability Threshold')
plt.ylabel('Metric Score')
plt.legend()
plt.show()

print(f"Optimal Decision Threshold for Maximum F1: {best_t:.2f} (F1 = {f1s[best_idx]:.4f})")

--- 
### 9. Feature Importance & Model Interpretability

In [ ]:
if hasattr(champion_model, 'feature_importances_'):
    importances = champion_model.feature_importances_
    feat_names = X_test_fe.columns
    fi_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances}).sort_values('Importance', ascending=False).head(15)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=fi_df['Importance'], y=fi_df['Feature'], palette='viridis')
    plt.title(f'Top 15 Feature Importances ({best_model_name})', fontsize=14, fontweight='bold')
    plt.xlabel('Relative Feature Importance')
    plt.show()

--- 
### 10. Conclusion & Deployment Next Steps
1. **Champion Model:** Tree-based ensemble with Cost-Sensitive Weighting achieved superior PR-AUC and F1-Score.
2. **Operational Tuning:** Selecting threshold `~0.30 - 0.40` prioritizes catching fraudulent activity while maintaining acceptable false alarm rates for customer operations.
3. **Production Deployment:** Exported artifacts (`best_model.joblib`, `preprocessor.joblib`, `feature_engineer.joblib`) power the live Streamlit interactive dashboard (`../app/app.py`).